In [1]:
import jLM
import h5py
import numpy as np


In [2]:
file_dir = "/data2/2024_Yeast_GS/my_current_code/rdme_ode_results"
file_name = "yeast_20240823_3_t60.0minGAE11.1mM.lm"
traj = h5py.File(file_dir + "/" + file_name, "r")

In [3]:
lattice_sites= np.array(traj['Model']['Diffusion']['LatticeSites'])
lattice_sites_names = traj['Parameters'].attrs['siteTypeNames'].decode().split(',')
print(lattice_sites_names)

['extracellular', 'obstructions', 'plasmaMembrane', 'cytoplasm', 'nucleoplasm', 'ribosomes']


In [4]:
species_names = np.array(traj['Parameters']['SpeciesNames'])
species_names = [name[0] for name in species_names]
print(species_names)


['DGrep', 'DGrep_G4d', 'DGrep_G4d_G80d', 'Rrep', 'Grep', 'DG1', 'DG1_G4d', 'DG1_G4d_G80d', 'R1', 'G1', 'DG2', 'DG2_G4d', 'DG2_G4d_G80d', 'R2', 'G2', 'DG3', 'DG3_G4d', 'DG3_G4d_G80d', 'R3', 'G3', 'G3i', 'DG4', 'R4', 'G4', 'G4d', 'DG80', 'DG80_G4d', 'DG80_G4d_G80d', 'R80', 'G80', 'G80d', 'G80d_G3i', 'ribosome', 'ribosomeR1', 'ribosomeR2', 'ribosomeR3', 'ribosomeR4', 'ribosomeR80', 'ribosomeGrep']


In [5]:
gene_species = {name: index+1 for index, name in enumerate(species_names) if name.startswith('D')}
print(gene_species)


{'DGrep': 1, 'DGrep_G4d': 2, 'DGrep_G4d_G80d': 3, 'DG1': 6, 'DG1_G4d': 7, 'DG1_G4d_G80d': 8, 'DG2': 11, 'DG2_G4d': 12, 'DG2_G4d_G80d': 13, 'DG3': 16, 'DG3_G4d': 17, 'DG3_G4d_G80d': 18, 'DG4': 22, 'DG80': 26, 'DG80_G4d': 27, 'DG80_G4d_G80d': 28}


reminder: the default location extract from the trajectory file is in the format of [z, y, x], 

In [6]:
lattice_first_frame = np.array(traj['Simulations']['0000001']['Lattice']['0000000001'])

# Print the shape of the lattice
print(f"Lattice shape: {lattice_first_frame.shape}")
print("The results will be saved as (x,y,z)")

# Iterate through gene species and find their locations in the lattice

def write_locations_to_file(filename, gene_data):
    with open(filename, 'w') as f:
        for gene_name, locations in gene_data.items():
            f.write(f"{gene_name}:\n")
            for loc in locations:
                f.write(f"  {','.join(map(str, loc))}\n")
            f.write("\n")

gene_locations_data = {}

for gene_name, gene_index in gene_species.items():
    gene_locations = np.argwhere(lattice_first_frame == gene_index)
    
    if gene_locations.size > 0:
        print(f"{gene_name} found at {len(gene_locations)} location(s):")
        gene_locations_data[gene_name] = []
        for location in gene_locations:
            # Reorder the coordinates from (z,y,x) to (x,y,z)
            reordered_location = location[:-1][::-1]
            print(f"  Position: {reordered_location}")
            gene_locations_data[gene_name].append(reordered_location)
    else:
        print(f"{gene_name} not found in the lattice")

# Save the data to a text file
output_filename = file_dir + "/" + file_name.split(".")[0] + "_gene_locations.txt"
write_locations_to_file(output_filename, gene_locations_data)
print(f"Gene locations have been saved to {output_filename}")

Lattice shape: (192, 192, 192, 16)
The results will be saved as (x,y,z)
DGrep not found in the lattice
DGrep_G4d not found in the lattice
DGrep_G4d_G80d found at 1 location(s):
  Position: [137  64 137]
DG1 not found in the lattice
DG1_G4d not found in the lattice
DG1_G4d_G80d found at 1 location(s):
  Position: [128  71 123]
DG2 not found in the lattice
DG2_G4d not found in the lattice
DG2_G4d_G80d found at 1 location(s):
  Position: [144  74 120]
DG3 not found in the lattice
DG3_G4d not found in the lattice
DG3_G4d_G80d found at 1 location(s):
  Position: [119  82 110]
DG4 found at 1 location(s):
  Position: [109  90 120]
DG80 not found in the lattice
DG80_G4d not found in the lattice
DG80_G4d_G80d found at 1 location(s):
  Position: [135  90  85]
Gene locations have been saved to /data2/2024_Yeast_GS/my_current_code/rdme_ode_results/yeast_20240823_3_t60_gene_locations.txt
